# 🥇 Gold Layer — Churn Features
**LushProtein Analytics | Medallion Architecture**

Reads from `medallion/gold/`, builds a customer-level churn feature table scoped to retail customers (excluding B2B/affiliate), enriches with survival analysis columns, first-order monetary features, discount classification, product features, and timing features, and writes to `medallion/gold/`.

| Source Tables | Gold Tables | Key Transformations |
|---|---|---|
| `gold_customer_orders`, `gold_customer_profiles`, `gold_first_order_products`, `gold_discount_analysis` | `gold_churn_features` | Survival event/duration columns, first-order monetary & discount features, product category flags, timing features, exclude B2B/affiliate |

## 0. Setup & Paths

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

BASE       = Path.cwd()
SILVER_DIR = BASE / 'medallion' / 'silver'
GOLD_DIR   = BASE / 'medallion' / 'gold'
GOLD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base   : {BASE}")
print(f"Silver : {SILVER_DIR}")
print(f"Gold   : {GOLD_DIR}")

Base   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505
Silver : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/silver
Gold   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/gold


---
## 1. Churn Features (`gold_customer_profiles` → `gold_churn_features`)

### Step 1 — Load all gold tables & build base

In [2]:
# Load all gold tables
co  = pd.read_parquet(GOLD_DIR / 'gold_customer_orders.parquet')
cp  = pd.read_parquet(GOLD_DIR / 'gold_customer_profiles.parquet')
fop = pd.read_parquet(GOLD_DIR / 'gold_first_order_products.parquet')
da  = pd.read_parquet(GOLD_DIR / 'gold_discount_analysis.parquet')

# Base — one row per customer from profiles
base = cp[[
    'customer_id', 'first_order_date', 'total_orders',
    'days_to_second_order', 'repeat_purchase_90d',
    'acquisition_channel', 'acquisition_country',
    'acquisition_discount_code', 'acquisition_discount_amt',
    'is_discount_acquired', 'rfm_group', 'rfm_score', 'gender'
]].copy()

### Step 2 — Survival analysis columns

In [3]:
# Survival analysis columns
# For survival analysis: event = made second order, duration = days to event
# Right-censored customers (no second order) get duration = days since first order
reference_date = pd.to_datetime(co['processed_at']).max()
base['first_order_date'] = pd.to_datetime(base['first_order_date'], utc=True)

base['event_occurred']   = base['days_to_second_order'].notna()  # True = made second order
base['survival_duration'] = base['days_to_second_order'].fillna(
    (reference_date - base['first_order_date']).dt.days
)
base['churned_before_90d'] = (
    base['days_to_second_order'].isna() |  # never repurchased
    (base['days_to_second_order'] > 90)     # repurchased but after 90 days
)

### Step 3 — First order monetary features

In [4]:
# First order monetary features
first_orders = co[co['is_first_order']].copy()

monetary = first_orders[[
    'customer_id', 'price_total', 'price_total_discount',
    'price_total_shipping', 'discount_code', 'discount_amount'
]].rename(columns={
    'price_total'          : 'first_order_aov',
    'price_total_discount' : 'first_order_total_discount',
    'price_total_shipping' : 'first_order_shipping',
    'discount_code'        : 'first_order_discount_code',
    'discount_amount'      : 'first_order_discount_amt',
})

# Discount percentage of order value
monetary['first_order_discount_pct'] = (
    monetary['first_order_discount_amt'] / monetary['first_order_aov'].replace(0, pd.NA)
).round(4)

monetary['first_order_discount_flag'] = monetary['first_order_discount_amt'].notna() & \
                                         (monetary['first_order_discount_amt'] > 0)

base = base.merge(monetary, on='customer_id', how='left')

### Step 4 — First order discount classification

In [5]:
# First order discount classification
first_order_discount = da[da['is_first_order']].copy()
base = base.merge(
    first_order_discount[['customer_id', 'discount_type',
                           'is_high_magnitude', 'is_b2b_or_affiliate']],
    on='customer_id', how='left'
)

### Step 5 — First order product features

In [6]:
# First order product features
# Number of line items and unique categories per first order
product_agg = fop.groupby('customer_id').agg(
    first_order_num_items      =('order_id', 'count'),
    first_order_num_categories =('product_category', 'nunique'),
    first_order_top_category   =('product_category', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    first_order_total_qty      =('quantity', 'sum'),
).reset_index()

# Boolean flags for product category presence
protein_cats = ['CLEAR', 'LEAN', 'PLT', 'SOY', 'BET', 'PRI', 'ELT']
for cat in ['ACC', 'BND', 'COL', 'CRE', 'CAP']:
    fop_cat = fop[fop['product_category'] == cat]['customer_id'].unique()
    product_agg[f'first_order_has_{cat.lower()}'] = product_agg['customer_id'].isin(fop_cat)

# Protein flag — any protein SKU in first order
fop_protein = fop[fop['product_category'].isin(protein_cats)]['customer_id'].unique()
product_agg['first_order_has_protein'] = product_agg['customer_id'].isin(fop_protein)

# Multi-category flag — bought more than one category
product_agg['first_order_is_multi_category'] = product_agg['first_order_num_categories'] > 1

base = base.merge(product_agg, on='customer_id', how='left')

### Step 6 — First order timing features

In [7]:
# First order timing features
first_orders['processed_at'] = pd.to_datetime(first_orders['processed_at'], utc=True)
first_orders['first_order_month']    = first_orders['processed_at'].dt.month
first_orders['first_order_quarter']  = first_orders['processed_at'].dt.quarter
first_orders['first_order_year']     = first_orders['processed_at'].dt.year
first_orders['first_order_dow']      = first_orders['processed_at'].dt.dayofweek  # 0=Mon, 6=Sun
first_orders['first_order_is_weekend'] = first_orders['first_order_dow'].isin([5, 6])

base = base.merge(
    first_orders[['customer_id', 'first_order_month', 'first_order_quarter',
                  'first_order_year', 'first_order_dow', 'first_order_is_weekend']],
    on='customer_id', how='left'
)


### Step 7 — Clean up, finalise & save

In [8]:
# Clean up and finalise

# Exclude B2B/affiliate from modelling — not retail customers
base = base[~base['is_b2b_or_affiliate'].fillna(False)].copy()
print(f"Rows after excluding B2B/affiliate: {len(base):,}")

# Flag zero-price first orders
base['first_order_is_zero_price'] = base['first_order_aov'] == 0

# Add subscriber flag via customer ID bridge
# Recharge and Shopify use different ID systems — bridge via shopify_order_id
bridge = pd.read_parquet(SILVER_DIR / 'silver_customer_id_bridge.parquet')
sub    = pd.read_parquet(GOLD_DIR / 'gold_subscription_behaviour.parquet')
sub_shopify_ids = set(
    bridge.merge(sub[['customer_id']], on='customer_id')['shopify_customer_id']
)
base['is_subscriber'] = base['customer_id'].isin(sub_shopify_ids)
print(f"Subscribers identified: {base['is_subscriber'].sum():,} / {len(base):,}")

gold_churn_features = base

print(f"\ngold_churn_features: {gold_churn_features.shape[0]:,} rows × {gold_churn_features.shape[1]} cols")
print(f"\nFeature summary:")
print(f"  Customers with event (2nd order): {gold_churn_features['event_occurred'].sum():,}")
print(f"  Customers right-censored:         {(~gold_churn_features['event_occurred']).sum():,}")
print(f"  Churned before 90d:               {gold_churn_features['churned_before_90d'].sum():,}")
print(f"  High magnitude discount:          {gold_churn_features['is_high_magnitude'].sum():,}")
print(f"  Has protein in first order:       {gold_churn_features['first_order_has_protein'].sum():,}")
print(f"  Multi-category first order:       {gold_churn_features['first_order_is_multi_category'].sum():,}")
print(f"\nTop category distribution:")
print(gold_churn_features['first_order_top_category'].value_counts(dropna=False).to_string())
print(f"\nSurvival duration stats (days):")
print(gold_churn_features['survival_duration'].describe().round(1).to_string())

gold_churn_features.to_parquet(GOLD_DIR / 'gold_churn_features.parquet', index=False)
print("\n✅ Saved: gold_churn_features.parquet")

Rows after excluding B2B/affiliate: 13,715
Subscribers identified: 548 / 13,715

gold_churn_features: 13,715 rows × 44 cols

Feature summary:
  Customers with event (2nd order): 4,492
  Customers right-censored:         9,223
  Churned before 90d:               10,710
  High magnitude discount:          738
  Has protein in first order:       11,516
  Multi-category first order:       5,218

Top category distribution:
first_order_top_category
BET      3802
CLEAR    1812
LEAN     1659
ACC      1652
PLT      1345
COL       701
BND       627
CAP       576
SOY       527
CRE       416
PRI       413
None      153
ELT        32

Survival duration stats (days):
count    13715.0
mean       668.7
std        704.1
min          0.0
25%         71.0
50%        321.0
75%       1343.0
max       2281.0

✅ Saved: gold_churn_features.parquet


#### Survival outcome

Customers with 2nd order (event): 4,492 (32.8%)

Right-censored (never repurchased): 9,223 (67.2%)

Event rate of 32.8% is reasonable for survival analysis — not too sparse

#### Churned before 90d: 10,710 (78.1%)

This combines customers who never repurchased (9,223) + those who repurchased but took longer than 90 days (1,487).

Means only 21.9% of customers converted within the critical 90-day window — the core challenge DS5 is trying to predict.

The 1,487 who eventually came back after 90 days are an interesting group — they're not lost, just slow. Worth studying separately in EDA.

#### Key features

High magnitude discount: 738 (5.4%) — core group for DS6 overlap, discount ≥30% off.

Has protein in first order: 11,516 (84.0%) — vast majority bought protein as expected.

Multi-category first order: 5,218 (38.1%) — 38% bought across more than one category, potentially a strong retention predictor for DS5's model.

#### Top category distribution

BET (3,802) — Better Whey leads after Lazada inclusion.

CLEAR (1,812) and LEAN (1,659) — dominant DTC categories.

ACC (1,652) — high accessories first purchase, interesting signal for Q1.

None (153, 1.1%) — negligible unclassifiable rows, very clean coverage.

#### Survival duration stats

Mean: 668.7 days, Median: 321 days — heavily right-skewed as expected for customer data.

25th percentile: 71 days — a quarter of observations have duration under 71 days, either fast repurchasers or very new customers.

Max: 2,281 days (~6.25 years) — your oldest customers from 2020, providing a long observation window for survival modelling.